# 02 — Follow a GPU experiment from launch to evidence

This notebook is the operational companion to Notebook 03. Read the saved outputs to
inspect verified work. It does not launch a GPU or repeat model inference.

## 1. See which stage each model has reached

A **smoke** checks one question. A **full pilot** covers all 16 supplied labels.
**Semantic scoring** then evaluates those saved answers with the pinned judge.
These are distinct milestones, so "Completed" in SageMaker alone is insufficient.

In [1]:
import json
from pathlib import Path

from IPython.display import HTML, display

from lava.evaluation.reporting import load_report
from lava.evaluation.walkthrough import TABLE_STYLE, comparison_tables, render_table, training_rates
from lava.notebook_support import find_repo_root
from lava.readers.runtime_logging import RuntimeEventLogger

ROOT = find_repo_root(Path.cwd())
logger = RuntimeEventLogger("notebook.verified_execution")
with logger.stage("01_verified_coverage", heartbeat_seconds=15):
    report = load_report(ROOT)
    pricing = json.loads((ROOT / "reports/aws/training_prices.json").read_text())
    tables = comparison_tables(report, training_rates(pricing))
    display(
        HTML(TABLE_STYLE + render_table(tables["coverage"], caption="Verified experiment stages"))
    )

{"component": "notebook.verified_execution", "elapsed_seconds": 0.0, "event": "01_verified_coverage.started", "level": "INFO", "stage_elapsed_seconds": 0.0, "timestamp_utc": "2026-09-08T03:17:29.516+00:00"}


Reader,Questions,Stage
Qwen3.5 · 4B,16 / 16,Full pilot scored
Qwen3.5 · 9B,16 / 16,Full pilot scored
Qwen3.8 · 27B NF4,16 / 16,Full pilot scored


{"component": "notebook.verified_execution", "elapsed_seconds": 0.425, "event": "01_verified_coverage.completed", "level": "INFO", "stage_elapsed_seconds": 0.424, "timestamp_utc": "2026-09-08T03:17:29.940+00:00"}


## 2. Understand the cloud lifecycle

| Stage | What is happening | What to watch |
|---|---|---|
| Pending | AWS is finding capacity | Queue elapsed time and server pending limit |
| Downloading | The instance downloads its container and inputs | SageMaker phase changes |
| Training | This job runs inference: weight loading, then questions | CloudWatch stage heartbeats and question counts |
| Uploading | AWS persists the final model-output artifacts | Upload completion |
| Completed | The process exited successfully | Artifact verification must still pass |

Here, SageMaker calls the service a *training job*, but our workload evaluates
frozen models; it does not fine-tune weights. Once provisioned, setup time can
be billable even before the first answer. Pending capacity time is separate.
Studio, S3 and CloudWatch have their own charges.

The current job contract allows 24 hours pending and one hour runtime. Those
limits live in AWS. Closing a browser or losing a notebook kernel does not stop
the managed job. A terminal monitor may disconnect; reconnect to the same job
instead of submitting another one.

In [2]:
with logger.stage("02_saved_run_history", heartbeat_seconds=15):
    rows = [
        {
            "Reader": run["label"],
            "Coverage": "Full pilot" if run["complete"] else "Smoke",
            "Questions": run["summary"]["record_count"],
            "Instance": run["instance_type"],
            "Billable time (s)": run["billable_seconds"],
            "Job": run["job_name"],
        }
        for run in report["runs"]
    ]
    display(HTML(render_table(rows, caption="Saved history · live status comes from the monitor")))

{"component": "notebook.verified_execution", "elapsed_seconds": 0.43, "event": "02_saved_run_history.started", "level": "INFO", "stage_elapsed_seconds": 0.0, "timestamp_utc": "2026-09-08T03:17:29.946+00:00"}


Reader,Coverage,Questions,Instance,Billable time (s),Job
Qwen3.5 · 4B,Smoke,1,ml.g5.2xlarge,336,lava-oracle-qwen35-4b-fused-direct-20260904193156
Qwen3.5 · 4B,Full pilot,16,ml.g5.2xlarge,380,lava-oracle-qwen35-4b-fused-direct-20260906180137
Qwen3.5 · 9B,Smoke,1,ml.g6e.2xlarge,350,lava-oracle-qwen35-9b-fused-direct-20260904203547
Qwen3.5 · 9B,Full pilot,16,ml.g6e.2xlarge,395,lava-oracle-qwen35-9b-fused-direct-20260906190503
Qwen3.8 · 27B NF4,Smoke,1,ml.g5.2xlarge,783,lava-oracle-qwen38-27b-nf4-g5-fused-direct-20260906044426
Qwen3.8 · 27B NF4,Full pilot,16,ml.g5.2xlarge,962,lava-oracle-qwen38-27b-nf4-g5-fused-direct-20260906233121


{"component": "notebook.verified_execution", "elapsed_seconds": 0.432, "event": "02_saved_run_history.completed", "level": "INFO", "stage_elapsed_seconds": 0.002, "timestamp_utc": "2026-09-08T03:17:29.948+00:00"}


## 3. Trace a run without repeating it

The tables above connect each completed pilot to its model, code revision, cloud job, and verified artifacts. Cloud status alone does not prove model quality: structured predictions and metrics are checked separately.

One operational interface handles preflight, launch, monitoring, artifact verification, and resume. Commands are documented in [the execution guide](../docs/benchmark.md) for optional reproduction. No new GPU run is needed to inspect this release.

## 4. Recover without discarding correct work

Each completed answer has a durable S3 checkpoint with its identity and checksums. A resume plan verifies which answers can be reused. A paid replacement attempt reuses those answers and does not silently relaunch a running or completed job. A crash before an answer's checkpoint is written may require that question again. Immutable judge decisions are also reused when scoring.

GitHub preserves code, public aggregates, and all five executed notebooks. S3 preserves private responses, checkpoints, and archived execution evidence. GPU scratch disks are working space; they are not the only copy of valuable results. CloudWatch carries runtime heartbeats, and saved notebook outputs show completed analysis steps.

Recovery tests exercise failed execution, corrupted/incompatible checkpoints, and interrupted notebook publication. [Notebook 04](04_evidence_retrieval.ipynb) also shows a measured fresh-process resume that reused every completed extraction and ranking.

[Research overview](00_reproducibility_and_protocol.ipynb) · [Model quality and cost](03_model_scaling_and_cost.ipynb)

In [3]:
logger.emit(
    "operations_walkthrough.completed",
    verified_runs=len(report["runs"]),
    cloud_resources_created=0,
)

{"cloud_resources_created": 0, "component": "notebook.verified_execution", "elapsed_seconds": 0.437, "event": "operations_walkthrough.completed", "level": "INFO", "timestamp_utc": "2026-09-08T03:17:29.952+00:00", "verified_runs": 6}
